# Uplift Modeling — Magnit Retail Dataset

**Задача:** оценить каузальный эффект промо-коммуникации на траты клиента (`rec_spend`) за 7 дней акции.  
**Метрика:** uplift@10 — нижняя граница 80% bootstrap CI разности средних между treatment и control в топ-10% по uplift score.  
**Данные:** ~355K клиентов, 89 признаков, ~90% нулей в таргете (zero-inflated continuous outcome).

---
```
Структура ноутбука:
  0. Установка и импорты
  1. Загрузка данных
  2. EDA и препроцессинг
  3. Метрика uplift@K с bootstrap
  4. Модели (T / S / X / DR / R / Hurdle)
  5. Сравнение и выбор ансамбля
  6. Санитарные проверки
  7. Финальный предикт
```

## 0. Установка и импорты

In [ ]:
# Установка пакетов (только если нужно)
# !pip install lightgbm catboost scikit-learn pandas numpy matplotlib seaborn -q

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold, train_test_split
from lightgbm import LGBMRegressor, LGBMClassifier
from lightgbm import early_stopping as lgb_es, log_evaluation as lgb_log
from catboost import CatBoostRegressor, CatBoostClassifier

warnings.filterwarnings('ignore')
os.environ['LOKY_MAX_CPU_COUNT'] = '4'
os.environ['OMP_NUM_THREADS']    = '4'

SEED = 42
np.random.seed(SEED)

pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')
plt.rcParams['figure.figsize'] = (13, 5)
sns.set_style('whitegrid')

DATA_DIR = '../data'
OUT_DIR  = 'results'
os.makedirs(OUT_DIR, exist_ok=True)

print('Импорты загружены')

## 1. Загрузка данных

In [ ]:
train = pd.read_parquet(os.path.join(DATA_DIR, 'train.parquet'))
test  = pd.read_parquet(os.path.join(DATA_DIR, 'test.parquet'))

print(f'train: {train.shape}  |  test: {test.shape}')
print(f'\nTreatment balance (train):')
print(train['treatment_flg'].value_counts(normalize=True).rename('share'))
print(f'\nTarget zeros: {(train["rec_spend"] == 0).mean():.1%}')
print(f'ATE: {train[train.treatment_flg==1].rec_spend.mean() - train[train.treatment_flg==0].rec_spend.mean():.4f}')

## 2. EDA и препроцессинг

Препроцессинг разделён на два шага:
- `fit_preprocess(train)` — обучается только на train, возвращает `stats` (медианы, категориальные маппинги)
- `apply_preprocess(df, stats)` — применяется к любому датасету (train или test)

Это гарантирует **отсутствие data leakage** между train и test.

In [ ]:
# ── Группы признаков по data_description ─────────────────────────────────────
TARGET    = 'rec_spend'
TREATMENT = 'treatment_flg'
DROP_COLS = ['user_id', 'treatment_flg', 'rec_spend']

# Категориальные (требуют Label Encoding)
CAT_COLS  = ['communication_type']

# Бинарные флаги (заполнять нулями при NaN)
FLAG_COLS = [
    'cus_mark_fav_cat_accept_flg_1_month_ago',
    'cus_mark_fav_cat_accept_flg_2_month_ago',
    'gendercalc'
]

print('Группы определены')

In [ ]:
# ── EDA: пропуски ─────────────────────────────────────────────────────────────
miss = train.isnull().mean().sort_values(ascending=False)
miss_nonzero = miss[miss > 0]
print(f'Признаков с пропусками: {len(miss_nonzero)} из {len(miss)}')
print(f'\nТоп-15 по доле пропусков:')
print(miss_nonzero.head(15).to_frame('missing_rate'))

# Визуализация
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
miss_nonzero.head(20).plot.barh(ax=axes[0], color='#5DCAA5')
axes[0].set_title('Топ-20 признаков по доле пропусков')
axes[0].set_xlabel('Доля пропусков')

# Распределение таргета (ненулевые)
nonzero = train[train[TARGET] > 0][TARGET]
axes[1].hist(np.log1p(nonzero), bins=50, color='#7F77DD', edgecolor='white', linewidth=0.3)
axes[1].set_title(f'log1p(rec_spend) для ненулевых\n(~{len(nonzero)/len(train):.1%} от всех клиентов)')
axes[1].set_xlabel('log1p(rec_spend)')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/eda_missing_and_target.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── EDA: проверка рандомизации ────────────────────────────────────────────────
# Если A/B тест чистый — признаки должны быть одинаково распределены в treatment и control
print('Проверка баланса treatment vs control по ключевым признакам:')
print(f'{"Признак":40s}  {"treatment":>10}  {"control":>10}  {"diff%":>8}')
print('-' * 72)
for col in ['age', 'rto', 'atv', 'n_trn', 'n_days_life']:
    if col in train.columns:
        t = train[train[TREATMENT]==1][col].mean()
        c = train[train[TREATMENT]==0][col].mean()
        diff_pct = abs(t - c) / (abs(c) + 1e-8) * 100
        flag = '⚠️' if diff_pct > 5 else '✓'
        print(f'{col:40s}  {t:10.3f}  {c:10.3f}  {diff_pct:7.2f}% {flag}')

In [ ]:
# ── Препроцессинг: fit ────────────────────────────────────────────────────────
def fit_preprocess(df: pd.DataFrame) -> dict:
    """Обучаем препроцессинг только на train. Возвращаем stats для apply."""
    stats = {}

    # 1. Медианы для числовых признаков (заполнение NaN)
    num_cols = [c for c in df.columns
                if c not in DROP_COLS + CAT_COLS + FLAG_COLS
                and df[c].dtype in ['float64', 'float32', 'int64', 'int32']]
    stats['num_medians'] = df[num_cols].median().to_dict()
    stats['num_cols']    = num_cols

    # 2. Label encoding для категориальных
    stats['cat_maps'] = {}
    for col in CAT_COLS:
        if col in df.columns:
            unique_vals = df[col].dropna().unique().tolist()
            stats['cat_maps'][col] = {v: i for i, v in enumerate(unique_vals)}

    # 3. Признаки с высокой долей пропусков (>50%) — запомним чтобы дропнуть
    miss_rate = df.isnull().mean()
    stats['high_miss_cols'] = miss_rate[miss_rate > 0.5].index.tolist()

    print(f'fit_preprocess: {len(num_cols)} числовых, {len(CAT_COLS)} категориальных')
    print(f'Признаков с >50% пропусков: {len(stats["high_miss_cols"])}')
    return stats


def apply_preprocess(df: pd.DataFrame, stats: dict) -> pd.DataFrame:
    """Применяем препроцессинг к любому датасету используя stats от fit."""
    df = df.copy()

    # 1. Числовые: заполняем медианой из train
    for col, median in stats['num_medians'].items():
        if col in df.columns:
            df[col] = df[col].fillna(median)

    # 2. Флаги: заполняем нулём
    for col in FLAG_COLS:
        if col in df.columns:
            df[col] = df[col].fillna(0)

    # 3. Категориальные: label encoding + -1 для неизвестных
    for col, mapping in stats['cat_maps'].items():
        if col in df.columns:
            df[col] = df[col].map(mapping).fillna(-1).astype(int)

    # 4. Дропаем признаки с высокой долей пропусков
    cols_to_drop = [c for c in stats['high_miss_cols'] if c in df.columns]
    if cols_to_drop:
        df = df.drop(columns=cols_to_drop)

    return df


# Применяем
preproc_stats = fit_preprocess(train)
train_proc    = apply_preprocess(train, preproc_stats)
test_proc     = apply_preprocess(test,  preproc_stats)

FEATURE_COLS = [c for c in train_proc.columns if c not in DROP_COLS]

X_full = train_proc[FEATURE_COLS].astype(float)
y_full = train_proc[TARGET].values
w_full = train_proc[TREATMENT].values
X_test = test_proc[FEATURE_COLS].astype(float)

print(f'\nX_full: {X_full.shape}')
print(f'X_test: {X_test.shape}')
print(f'NaN остаток: {X_full.isnull().sum().sum()}')

## 3. Метрика uplift@K с bootstrap

**uplift@K** = разность средних `rec_spend` между treatment и control в топ-K% клиентов по uplift score.

Используем нижнюю границу 80% bootstrap CI — это штрафует нестабильные модели.

In [ ]:
def uplift_at_k(y, w, scores, k=0.10, n_boot=300, ci=0.80, seed=SEED):
    """uplift@K с bootstrap CI. Возвращает (point, lower_ci, upper_ci)."""
    df   = pd.DataFrame({'y': y, 'w': w, 's': scores}).sort_values('s', ascending=False)
    top  = df.iloc[:int(len(df) * k)].reset_index(drop=True)
    n    = len(top)
    point = top[top.w==1]['y'].mean() - top[top.w==0]['y'].mean()

    rng   = np.random.RandomState(seed)
    boots = []
    for _ in range(n_boot):
        samp = top.iloc[rng.choice(n, n, replace=True)]
        boots.append(samp[samp.w==1]['y'].mean() - samp[samp.w==0]['y'].mean())

    alpha = (1 - ci) / 2
    return (round(point, 4),
            round(np.percentile(boots, alpha * 100), 4),
            round(np.percentile(boots, (1-alpha) * 100), 4))


def cv_evaluate(model_fn, name, n_folds=3, fast=True):
    """
    Кросс-валидация модели.
    fast=True: CV на 100K сэмпле, bootstrap n=100 (для быстрой оценки при подборе).
    fast=False: CV на полных данных, bootstrap n=300 (для финального сравнения).
    """
    n_boot = 100 if fast else 300
    X_cv   = X_full.sample(n=min(100_000, len(X_full)), random_state=SEED) if fast else X_full
    y_cv   = y_full[X_cv.index]
    w_cv   = w_full[X_cv.index]
    X_cv   = X_cv.reset_index(drop=True)

    kf = KFold(n_splits=n_folds, shuffle=True, random_state=SEED)
    pts, los = [], []

    for fold, (tr_idx, va_idx) in enumerate(kf.split(X_cv), 1):
        Xtr, Xva = X_cv.iloc[tr_idx], X_cv.iloc[va_idx]
        ytr, yva = y_cv[tr_idx], y_cv[va_idx]
        wtr, wva = w_cv[tr_idx], w_cv[va_idx]

        Xt = Xtr[wtr==1].reset_index(drop=True); yt = ytr[wtr==1]
        Xc = Xtr[wtr==0].reset_index(drop=True); yc = ytr[wtr==0]

        scores = model_fn(Xt, yt, Xc, yc, Xva)
        pt, lo, _ = uplift_at_k(yva, wva, scores, n_boot=n_boot)
        pts.append(pt); los.append(lo)
        print(f'  fold {fold}: point={pt:.4f}  lower={lo:.4f}')

    pt_m = round(np.mean(pts), 4)
    lo_m = round(np.mean(los), 4)
    print(f'  [{name}] CV point={pt_m}  CV lower={lo_m}\n')
    return pt_m, lo_m


results = {}  # {name: (point, lower_ci)}
print('Метрика и CV готовы')

## 4. Модели

### Параметры и хелперы для обучения

In [ ]:
# ── Параметры моделей (оптимизированы для скорости + качества) ───────────────
LGB_P = dict(
    n_estimators=200, learning_rate=0.05, num_leaves=31,
    min_child_samples=100, subsample=0.8, colsample_bytree=0.8,
    random_state=SEED, n_jobs=4, verbose=-1
)
CB_P = dict(
    iterations=200, learning_rate=0.05, depth=5, l2_leaf_reg=5.0,
    random_seed=SEED, verbose=0, thread_count=4
)
LGB_CLF_P = dict(
    n_estimators=150, learning_rate=0.05, num_leaves=31,
    min_child_samples=100, scale_pos_weight=9,
    random_state=SEED, n_jobs=4, verbose=-1
)
CB_CLF_P = dict(
    iterations=150, learning_rate=0.05, depth=4, l2_leaf_reg=5.0,
    class_weights={0: 1, 1: 9},
    random_seed=SEED, verbose=0, thread_count=4
)


def fit_lgb(params, X_tr, y_tr, es_frac=0.15):
    """LightGBM с early stopping."""
    n   = len(X_tr)
    cut = max(int(n * es_frac), 200)
    p   = {**params, 'n_estimators': 1000}
    return LGBMRegressor(**p).fit(
        X_tr.iloc[:-cut], y_tr[:-cut],
        eval_set=[(X_tr.iloc[-cut:], y_tr[-cut:])],
        callbacks=[lgb_es(50), lgb_log(-1)]
    )


def fit_cb(params, X_tr, y_tr, es_frac=0.15, clf=False):
    """CatBoost с early stopping."""
    n   = len(X_tr)
    cut = max(int(n * es_frac), 200)
    p   = {**params, 'iterations': 1000, 'early_stopping_rounds': 50}
    Model = CatBoostClassifier if clf else CatBoostRegressor
    return Model(**p).fit(
        X_tr.iloc[:-cut], y_tr[:-cut],
        eval_set=(X_tr.iloc[-cut:], y_tr[-cut:]),
        verbose=0
    )


print('Параметры и хелперы готовы')

### 4.1 T-learner (Ridge baseline)

In [ ]:
def t_ridge(Xt, yt, Xc, yc, Xv):
    """T-learner с Ridge регрессией. Быстрый baseline."""
    pipe_t = make_pipeline(SimpleImputer(strategy='constant', fill_value=0),
                           StandardScaler(), Ridge(alpha=10.0))
    pipe_c = make_pipeline(SimpleImputer(strategy='constant', fill_value=0),
                           StandardScaler(), Ridge(alpha=10.0))
    pipe_t.fit(Xt, yt)
    pipe_c.fit(Xc, yc)
    return pipe_t.predict(Xv) - pipe_c.predict(Xv)

pt, lo = cv_evaluate(t_ridge, 'T-Ridge')
results['T-Ridge'] = (pt, lo)

### 4.2 T-learner (LightGBM)

In [ ]:
def t_lgb(Xt, yt, Xc, yc, Xv):
    """T-learner с LightGBM."""
    mt = fit_lgb(LGB_P, Xt, yt)
    mc = fit_lgb(LGB_P, Xc, yc)
    return mt.predict(Xv) - mc.predict(Xv)

pt, lo = cv_evaluate(t_lgb, 'T-LGB')
results['T-LGB'] = (pt, lo)

### 4.3 S-learner (CatBoost)

In [ ]:
def s_cb(Xt, yt, Xc, yc, Xv):
    """S-learner: одна модель, treatment как признак."""
    Xtr = pd.concat([Xt, Xc]).copy()
    Xtr['_w'] = np.concatenate([np.ones(len(yt)), np.zeros(len(yc))])
    ytr = np.concatenate([yt, yc])
    m   = fit_cb(CB_P, Xtr, ytr)
    Xv1 = Xv.copy(); Xv1['_w'] = 1.0
    Xv0 = Xv.copy(); Xv0['_w'] = 0.0
    return m.predict(Xv1) - m.predict(Xv0)

pt, lo = cv_evaluate(s_cb, 'S-CatBoost')
results['S-CatBoost'] = (pt, lo)

### 4.4 X-learner (CatBoost)

In [ ]:
def x_cb(Xt, yt, Xc, yc, Xv):
    """X-learner: двухступенчатая импутация treatment effect.
    Эффективен при дисбалансе групп."""
    mt    = fit_cb(CB_P, Xt, yt)
    mc    = fit_cb(CB_P, Xc, yc)
    tau_t = fit_cb(CB_P, Xt, yt - mc.predict(Xt))
    tau_c = fit_cb(CB_P, Xc, mt.predict(Xc) - yc)
    p     = len(yt) / (len(yt) + len(yc))  # доля treated
    return p * tau_t.predict(Xv) + (1 - p) * tau_c.predict(Xv)

pt, lo = cv_evaluate(x_cb, 'X-CatBoost')
results['X-CatBoost'] = (pt, lo)

### 4.5 DR-learner (CatBoost) ← лучшая модель на хакатоне

In [ ]:
def dr_cb(Xt, yt, Xc, yc, Xv, n_folds=3):
    """DR-learner (doubly robust).
    Состоятелен если хотя бы одна из двух nuisance функций
    (outcome model или propensity score) оценена корректно."""
    Xd = pd.concat([Xt, Xc]).reset_index(drop=True)
    yd = np.concatenate([yt, yc])
    wd = np.concatenate([np.ones(len(yt)), np.zeros(len(yc))])
    psi = np.zeros(len(Xd))  # DR pseudo-outcomes

    kf = KFold(n_splits=n_folds, shuffle=True, random_state=SEED)
    for tr_idx, val_idx in kf.split(Xd):
        Xtr, Xval = Xd.iloc[tr_idx], Xd.iloc[val_idx]
        ytr, yval = yd[tr_idx], yd[val_idx]
        wtr, wval = wd[tr_idx], wd[val_idx]

        # Propensity score
        ps = LGBMClassifier(n_estimators=200, learning_rate=0.05, num_leaves=31,
                            min_child_samples=100, random_state=SEED, n_jobs=4, verbose=-1)
        ps.fit(Xtr, wtr)
        e = ps.predict_proba(Xval)[:, 1].clip(0.05, 0.95)

        # Outcome models
        mu1 = fit_cb(CB_P, Xtr[wtr==1].reset_index(drop=True), ytr[wtr==1])
        mu0 = fit_cb(CB_P, Xtr[wtr==0].reset_index(drop=True), ytr[wtr==0])

        # DR pseudo-outcome: psi = (mu1 - mu0) + W*(Y-mu1)/e - (1-W)*(Y-mu0)/(1-e)
        psi[val_idx] = (
            mu1.predict(Xval) - mu0.predict(Xval)
            + wval  * (yval - mu1.predict(Xval)) / e
            - (1 - wval) * (yval - mu0.predict(Xval)) / (1 - e)
        )

    # Финальная регрессия pseudo-outcomes на признаки
    return fit_cb(CB_P, Xd, psi).predict(Xv)

pt, lo = cv_evaluate(dr_cb, 'DR-CatBoost')
results['DR-CatBoost'] = (pt, lo)

### 4.6 R-learner (CatBoost)

In [ ]:
def r_cb(Xt, yt, Xc, yc, Xv):
    """R-learner (Robinson decomposition).
    tau* = argmin E[(Y - m(X) - (W - e(X))*tau(X))^2]
    Имеет quasi-oracle свойство: скорость сходимости
    зависит только от сложности tau*, не от m* и e*."""
    Xd = pd.concat([Xt, Xc]).reset_index(drop=True)
    yd = np.concatenate([yt, yc])
    wd = np.concatenate([np.ones(len(yt)), np.zeros(len(yc))])

    # E[Y|X] и E[W|X]
    m_y = fit_cb(CB_P, Xd, yd)
    ps  = LGBMClassifier(n_estimators=200, learning_rate=0.05, num_leaves=31,
                         min_child_samples=100, random_state=SEED, n_jobs=4, verbose=-1)
    ps.fit(Xd, wd)
    e = ps.predict_proba(Xd)[:, 1].clip(0.05, 0.95)

    # R-loss residuals
    Y_res  = yd - m_y.predict(Xd)
    W_res  = wd - e
    pseudo = Y_res / (W_res + 1e-8)  # псевдо-outcome
    sw     = W_res ** 2               # sample weights

    # Регрессия pseudo-outcome с весами
    n = len(Xd); cut = max(int(n * 0.15), 200)
    p = {**CB_P, 'iterations': 1000, 'early_stopping_rounds': 50}
    m = CatBoostRegressor(**p).fit(
        Xd.iloc[:-cut], pseudo[:-cut],
        eval_set=(Xd.iloc[-cut:], pseudo[-cut:]),
        sample_weight=sw[:-cut],
        verbose=0
    )
    return m.predict(Xv)

pt, lo = cv_evaluate(r_cb, 'R-CatBoost')
results['R-CatBoost'] = (pt, lo)

### 4.7 Hurdle (CatBoost) — специально для zero-inflated outcome

In [ ]:
def hurdle_cb(Xt, yt, Xc, yc, Xv):
    """Hurdle model: P(Y>0) × E[Y|Y>0] отдельно для treatment и control.
    Специально для ~90% нулей в rec_spend.
    uplift = P_t(Y>0)*E_t[Y|Y>0] - P_c(Y>0)*E_c[Y|Y>0]"""
    # Классификаторы: купит / не купит
    clf_t = fit_cb(CB_CLF_P, Xt, (yt > 0).astype(int), clf=True)
    clf_c = fit_cb(CB_CLF_P, Xc, (yc > 0).astype(int), clf=True)

    # Регрессоры: сколько потратит если купит
    reg_t = fit_cb(CB_P, Xt[yt > 0].reset_index(drop=True), yt[yt > 0])
    reg_c = fit_cb(CB_P, Xc[yc > 0].reset_index(drop=True), yc[yc > 0])

    p_t = clf_t.predict_proba(Xv)[:, 1]
    p_c = clf_c.predict_proba(Xv)[:, 1]

    return p_t * reg_t.predict(Xv) - p_c * reg_c.predict(Xv)

pt, lo = cv_evaluate(hurdle_cb, 'Hurdle-CatBoost')
results['Hurdle-CatBoost'] = (pt, lo)

## 5. Сравнение моделей и выбор ансамбля

In [ ]:
# ── Сводная таблица результатов CV ────────────────────────────────────────────
results_df = pd.DataFrame(
    [(name, pt, lo) for name, (pt, lo) in results.items()],
    columns=['model', 'cv_point', 'cv_lower_ci']
).sort_values('cv_lower_ci', ascending=False)

print('\nРезультаты CV (сортировка по lower CI):')
print(results_df.to_string(index=False))

# Сохраняем
results_df.to_csv(f'{OUT_DIR}/cv_results.csv', index=False)

# Визуализация
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#7F77DD' if i == 0 else '#B4B2A9' for i in range(len(results_df))]
bars = ax.barh(results_df['model'], results_df['cv_lower_ci'], color=colors)
ax.axvline(0, color='black', linewidth=0.8, linestyle='--', alpha=0.4)
ax.set_xlabel('uplift@10 lower 80% CI (CV)')
ax.set_title('Сравнение моделей по нижней границе bootstrap CI')
for bar, val in zip(bars, results_df['cv_lower_ci']):
    ax.text(val + 0.05, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=10)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/model_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── Ансамбль топ-3 моделей по lower CI ───────────────────────────────────────
# Выбираем топ-3 по CV lower CI
top3 = results_df.head(3)['model'].tolist()
print(f'Топ-3 модели для ансамбля: {top3}')

MODEL_FNS = {
    'T-Ridge':       t_ridge,
    'T-LGB':         t_lgb,
    'S-CatBoost':    s_cb,
    'X-CatBoost':    x_cb,
    'DR-CatBoost':   dr_cb,
    'R-CatBoost':    r_cb,
    'Hurdle-CatBoost': hurdle_cb,
}

# Веса: обратно пропорционально gap между train и holdout
# (модели с меньшим переобучением получают больший вес)
X_inner, X_hold, y_inner, y_hold, w_inner, w_hold = train_test_split(
    X_full, y_full, w_full, test_size=0.20, random_state=271
)
X_inner = X_inner.reset_index(drop=True)
X_hold  = X_hold.reset_index(drop=True)

Xt_inner = X_inner[w_inner==1].reset_index(drop=True); yt_inner = y_inner[w_inner==1]
Xc_inner = X_inner[w_inner==0].reset_index(drop=True); yc_inner = y_inner[w_inner==0]

print(f'\n{"Модель":20s}  {"train_lo":>9}  {"hold_lo":>9}  {"gap":>7}')
print('-' * 52)
gaps = {}
for name in top3:
    fn = MODEL_FNS[name]
    sc_tr = fn(Xt_inner, yt_inner, Xc_inner, yc_inner, X_inner)
    sc_ho = fn(Xt_inner, yt_inner, Xc_inner, yc_inner, X_hold)
    _, lt, _ = uplift_at_k(y_inner, w_inner, sc_tr, n_boot=100)
    _, lh, _ = uplift_at_k(y_hold,  w_hold,  sc_ho, n_boot=100)
    gap = lt - lh
    gaps[name] = gap
    print(f'{name:20s}  {lt:9.4f}  {lh:9.4f}  {gap:7.4f}')

# Веса обратно пропорциональны gap
inv_gaps = {m: 1.0 / (g + 0.1) for m, g in gaps.items()}
total    = sum(inv_gaps.values())
weights  = {m: v / total for m, v in inv_gaps.items()}
print(f'\nВеса ансамбля:')
for m, w in weights.items():
    print(f'  {m}: {w:.3f}')

## 6. Санитарные проверки

In [ ]:
# Финальный предикт ансамбля на holdout для проверок
Xt_f = X_full[w_full==1].reset_index(drop=True); yt_f = y_full[w_full==1]
Xc_f = X_full[w_full==0].reset_index(drop=True); yc_f = y_full[w_full==0]

# Предикты на holdout от каждой модели топ-3
preds_hold = {}
for name in top3:
    preds_hold[name] = MODEL_FNS[name](
        Xt_inner, yt_inner, Xc_inner, yc_inner, X_hold
    )
ensemble_hold = sum(weights[m] * preds_hold[m] for m in top3)

print('=' * 50)
print('САНИТАРНЫЕ ПРОВЕРКИ')
print('=' * 50)

# 1. Корреляции между моделями
print('\n1. Корреляции между моделями в топ-3 (на holdout):')
names = list(top3)
for i in range(len(names)):
    for j in range(i+1, len(names)):
        r = np.corrcoef(preds_hold[names[i]], preds_hold[names[j]])[0, 1]
        flag = '⚠️ высокая корреляция' if r > 0.95 else '✓'
        print(f'   {names[i]} vs {names[j]}: r={r:.3f}  {flag}')

# 2. Доля клиентов с отрицательным uplift (sleeping dogs)
neg_share = (ensemble_hold < 0).mean()
print(f'\n2. Доля клиентов с отрицательным uplift: {neg_share:.1%}')
print(f'   (это sleeping dogs — промо снижает их траты)')

# 3. Стабильность топ-10% через bootstrap
n_h     = len(ensemble_hold)
top_k   = int(n_h * 0.1)
top_set = set(np.argsort(ensemble_hold)[-top_k:])
rng     = np.random.RandomState(271)
overlaps = []
for _ in range(300):
    idx = rng.choice(n_h, n_h, replace=True)
    top_b = set(idx[np.argsort(ensemble_hold[idx])[-top_k:]])
    overlaps.append(len(top_set & top_b) / top_k)
stability = np.mean(overlaps)
expected  = 1 - (1 - 1/n_h) ** n_h
efficiency = stability / expected
print(f'\n3. Стабильность топ-10% (bootstrap n=300):')
print(f'   overlap={stability:.3f}  потолок≈{expected:.3f}  эффективность={efficiency:.3f}')
flag = '✓' if efficiency > 0.95 else '⚠️ нестабильное ранжирование'
print(f'   {flag}')

# 4. Ожидаемый лидерборд
_, lo_hold, _ = uplift_at_k(y_hold, w_hold, ensemble_hold, n_boot=300)
print(f'\n4. Ожидаемый uplift@10 lower CI на лидерборде: {lo_hold:.4f}')

## 7. Финальный предикт

In [ ]:
# Обучаем топ-3 на полном train, предсказываем test
print('Обучаем ансамбль на полном train...')
final_preds = {}
for name in top3:
    print(f'  {name}...')
    final_preds[name] = MODEL_FNS[name](Xt_f, yt_f, Xc_f, yc_f, X_test)

final_scores = sum(weights[m] * final_preds[m] for m in top3)

# Сохраняем predictions.csv
submission = pd.DataFrame({
    'user_id':     test_proc['user_id'].values,
    'UPLIFT_SCORE': final_scores
})
submission.to_csv(f'{OUT_DIR}/predictions.csv', index=False)

print(f'\nСохранено: {len(submission)} строк')
print(f'mean={final_scores.mean():.4f}  std={final_scores.std():.4f}')
print(f'p10={np.percentile(final_scores,10):.4f}  p90={np.percentile(final_scores,90):.4f}')
print(f'\nПервые 5 строк:')
print(submission.head())

In [ ]:
# Визуализация финального распределения uplift scores
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(final_scores, bins=60, color='#7F77DD', edgecolor='white', linewidth=0.3)
axes[0].axvline(0, color='#D85A30', linewidth=1.5, linestyle='--', label='uplift=0')
axes[0].set_title('Распределение uplift scores (test)')
axes[0].set_xlabel('Uplift score')
axes[0].legend()

# Qini curve на holdout
df_q = pd.DataFrame({'y': y_hold, 'w': w_hold, 's': ensemble_hold})\
         .sort_values('s', ascending=False).reset_index(drop=True)
n_q  = len(df_q)
cumulative_uplift = []
for k in range(1, n_q + 1):
    top = df_q.iloc[:k]
    if top['w'].sum() > 0 and (1 - top['w']).sum() > 0:
        u = top[top.w==1]['y'].mean() - top[top.w==0]['y'].mean()
    else:
        u = 0
    cumulative_uplift.append(u)

x_axis = np.arange(1, n_q + 1) / n_q * 100
axes[1].plot(x_axis, cumulative_uplift, color='#7F77DD', linewidth=1.5, label='Ensemble')
axes[1].axhline(cumulative_uplift[-1], color='gray', linewidth=1, linestyle='--', label='Random')
axes[1].axvline(10, color='#D85A30', linewidth=1, linestyle=':', alpha=0.7, label='Top-10%')
axes[1].set_title('Uplift curve (holdout)')
axes[1].set_xlabel('% клиентов (по убыванию uplift score)')
axes[1].set_ylabel('uplift@K')
axes[1].legend()

plt.tight_layout()
plt.savefig(f'{OUT_DIR}/final_scores_and_qini.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'\nВсе файлы сохранены в {OUT_DIR}/')
print(f'  predictions.csv      — финальный сабмит')
print(f'  cv_results.csv       — результаты CV всех моделей')
print(f'  model_comparison.png — сравнение моделей')
print(f'  final_scores_and_qini.png — финальные графики')